# Breast Cancer Big Data and Machine Learning Project

## Team Version: 4-Part Organized Notebook

This notebook is divided into four clear parts so each team member can explain one section during the discussion.

- **Part 1:** Project setup and dataset understanding
- **Part 2:** Data analysis and visualization using Pandas
- **Part 3:** Big Data analysis using PySpark
- **Part 4:** Machine learning, evaluation, and conclusion

The code is written in a direct and simple style without custom functions, custom classes, or sklearn pipelines.


## Team Work Distribution

| Team Member | Main Responsibility |
|---|---|
| Member 1 | Project setup and dataset understanding |
| Member 2 | Data analysis and visualization |
| Member 3 | PySpark and Big Data operations |
| Member 4 | Machine learning, evaluation, and final conclusion |


# PART 1: Project Setup and Dataset Understanding

**Responsible member:** Member 1  
This part explains the project idea, imports the libraries, loads the dataset, and checks the basic structure of the data.


## 1.1 Project Overview

The project applies data analysis, Big Data concepts, and machine learning to a medical dataset.  
The main goal is to analyze breast cancer diagnostic features and build a model that predicts whether a tumor is benign or malignant.


## 1.2 Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay


## 1.3 Load Dataset


In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})

df.head()


## 1.4 Dataset Shape


In [ ]:
rows, columns = df.shape
print("Number of rows:", rows)
print("Number of columns:", columns)


## 1.5 Dataset Columns


In [ ]:
df.columns.tolist()


## 1.6 Data Types


In [ ]:
df.dtypes


## 1.7 Missing Values Check


In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)


## 1.8 Target Distribution


In [ ]:
df["target_name"].value_counts()


## Part 1 Insight

The dataset has numerical medical features and a clear target column.  
There are no missing values, so the data is ready for analysis and machine learning.


# PART 2: Data Analysis and Visualization Using Pandas

**Responsible member:** Member 2  
This part analyzes the data using Pandas and visualizations to extract useful insights.


## 2.1 Descriptive Statistics


In [ ]:
df.describe().T.head(15)


## 2.2 Separate Numerical Features


In [ ]:
feature_columns = data.feature_names.tolist()
numeric_df = df[feature_columns + ["target"]]

numeric_df.head()


## 2.3 Target Count Plot


In [ ]:
df["target_name"].value_counts().plot(kind="bar", figsize=(6, 4))
plt.title("Target Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 2.4 Mean Values by Diagnosis


In [ ]:
df.groupby("target_name")[feature_columns].mean().T.head(15)


## 2.5 Correlation with Target


In [ ]:
corr_with_target = numeric_df.corr()["target"].drop("target").sort_values(key=abs, ascending=False)
corr_with_target.head(15)


## 2.6 Top Correlated Features Bar Chart


In [ ]:
corr_with_target.head(10).plot(kind="barh", figsize=(8, 5))
plt.title("Top Features Correlated with Target")
plt.xlabel("Correlation")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 2.7 Heatmap for Strongest Features


In [ ]:
top_features = corr_with_target.head(8).index.tolist()
heatmap_data = df[top_features + ["target"]].corr()

plt.figure(figsize=(8, 6))
plt.imshow(heatmap_data, aspect="auto")
plt.colorbar()
plt.xticks(range(len(heatmap_data.columns)), heatmap_data.columns, rotation=90)
plt.yticks(range(len(heatmap_data.index)), heatmap_data.index)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## 2.8 Histogram Example


In [ ]:
df["mean radius"].plot(kind="hist", bins=30, figsize=(6, 4))
plt.title("Distribution of Mean Radius")
plt.xlabel("Mean Radius")
plt.tight_layout()
plt.show()


## 2.9 Boxplot Example


In [ ]:
df.boxplot(column="mean radius", by="target_name", figsize=(6, 4))
plt.title("Mean Radius by Diagnosis")
plt.suptitle("")
plt.xlabel("Diagnosis")
plt.ylabel("Mean Radius")
plt.tight_layout()
plt.show()


## 2.10 Scatter Plot Example


In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df["mean radius"], df["mean texture"], c=df["target"])
plt.xlabel("Mean Radius")
plt.ylabel("Mean Texture")
plt.title("Mean Radius vs Mean Texture")
plt.tight_layout()
plt.show()


## 2.11 Filtering Example


In [ ]:
large_radius_cases = df[df["mean radius"] > df["mean radius"].mean()]
large_radius_cases[["mean radius", "mean texture", "target_name"]].head()


## 2.12 Aggregation Example


In [ ]:
df.groupby("target_name").agg({
    "mean radius": ["mean", "min", "max"],
    "mean texture": ["mean", "min", "max"],
    "mean area": ["mean", "min", "max"]
})


## Part 2 Insights

- Some medical features have a strong relationship with the target.
- The average values of important features differ between benign and malignant cases.
- Correlation and visualization help identify the strongest features before modeling.


# PART 3: Big Data Analysis Using PySpark

**Responsible member:** Member 3  
This part uses PySpark DataFrames to apply Big Data-style operations such as filtering, grouping, aggregation, and Spark SQL.


## 3.1 Install and Import PySpark


In [ ]:
try:
    import pyspark
except ImportError:
    !pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier as SparkRandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator


## 3.2 Create Spark Session


In [ ]:
spark = SparkSession.builder.appName("BreastCancerBigDataProject").getOrCreate()
spark


## 3.3 Convert Pandas DataFrame to Spark DataFrame


In [ ]:
spark_df = spark.createDataFrame(df)
spark_df.show(5)


## 3.4 Spark Schema


In [ ]:
spark_df.printSchema()


## 3.5 Count Rows and Columns in Spark


In [ ]:
print("Rows:", spark_df.count())
print("Columns:", len(spark_df.columns))


## 3.6 PySpark Filtering


In [ ]:
spark_df.filter(F.col("mean radius") > 15).select(
    "mean radius", "mean texture", "target", "target_name"
).show(10)


## 3.7 PySpark Grouping


In [ ]:
spark_df.groupBy("target_name").count().show()


## 3.8 PySpark Aggregation


In [ ]:
spark_df.groupBy("target_name").agg(
    F.round(F.avg("mean radius"), 2).alias("avg_mean_radius"),
    F.round(F.avg("mean texture"), 2).alias("avg_mean_texture"),
    F.round(F.avg("mean area"), 2).alias("avg_mean_area")
).show()


## 3.9 Spark SQL


In [ ]:
spark_df.createOrReplaceTempView("cancer_data")

spark.sql("""
SELECT target_name,
       COUNT(*) AS cases,
       ROUND(AVG(`mean radius`), 2) AS avg_radius,
       ROUND(AVG(`mean texture`), 2) AS avg_texture
FROM cancer_data
GROUP BY target_name
""").show()


## 3.10 PySpark Correlation with Target


In [ ]:
spark_corr = []

for feature in feature_columns:
    value = spark_df.stat.corr(feature, "target")
    spark_corr.append([feature, value, abs(value)])

spark_corr_df = pd.DataFrame(spark_corr, columns=["Feature", "Correlation", "Abs_Correlation"])
spark_corr_df.sort_values("Abs_Correlation", ascending=False).head(15)


## 3.11 Prepare Spark Data for Machine Learning


In [ ]:
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
spark_ml_df = assembler.transform(spark_df).select("features", F.col("target").alias("label"))

spark_train, spark_test = spark_ml_df.randomSplit([0.8, 0.2], seed=42)

spark_ml_df.show(5, truncate=False)


## 3.12 Train PySpark Random Forest Model


In [ ]:
spark_rf = SparkRandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

spark_model = spark_rf.fit(spark_train)
spark_predictions = spark_model.transform(spark_test)

spark_predictions.select("label", "prediction", "probability").show(10, truncate=False)


## 3.13 Evaluate PySpark Model


In [ ]:
spark_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(spark_predictions)

spark_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(spark_predictions)

print("PySpark Accuracy:", round(spark_accuracy, 4))
print("PySpark ROC-AUC:", round(spark_auc, 4))


## 3.14 PySpark Feature Importance


In [ ]:
spark_importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": spark_model.featureImportances.toArray()
})

spark_importance_df.sort_values("Importance", ascending=False).head(15)


## Part 3 Insights

- PySpark was used to apply Big Data-style processing.
- Spark DataFrames support filtering, grouping, aggregation, and SQL queries.
- Spark MLlib was used to train and evaluate a Random Forest model.


# PART 4: Machine Learning, Evaluation, and Conclusion

**Responsible member:** Member 4  
This part trains a direct machine learning model using sklearn, evaluates the model, and explains the final results.


## 4.1 Prepare Features and Target


In [ ]:
X = df[feature_columns]
y = df["target"]

X.head()


## 4.2 Train Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 4.3 Feature Scaling


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 4.4 Train Random Forest Model


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train_scaled, y_train)


## 4.5 Model Prediction


In [ ]:
y_pred = model.predict(X_test_scaled)
y_score = model.predict_proba(X_test_scaled)[:, 1]

y_pred[:10]


## 4.6 Evaluation Metrics


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_score)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

results


## 4.7 Classification Report


In [ ]:
print(classification_report(y_test, y_pred, target_names=data.target_names))


## 4.8 Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], data.target_names)
plt.yticks([0, 1], data.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


## 4.9 ROC Curve


In [ ]:
RocCurveDisplay.from_predictions(y_test, y_score)
plt.title("ROC Curve")
plt.tight_layout()
plt.show()


## 4.10 Feature Importance


In [ ]:
importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values("Importance", ascending=False)
importance_df.head(15)


## 4.11 Feature Importance Chart


In [ ]:
importance_df.head(10).plot(
    x="Feature",
    y="Importance",
    kind="barh",
    figsize=(8, 5),
    legend=False
)
plt.title("Top 10 Important Features")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 4.12 Final Model Result


In [ ]:
print("Final Model: Random Forest Classifier")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))


## Final Conclusion

This project successfully applied data analysis, Big Data concepts, and machine learning on the breast cancer dataset.

- Pandas was used for data understanding, visualization, and statistical analysis.
- PySpark was used for Big Data-style processing including filtering, grouping, aggregation, SQL, and Spark MLlib.
- Random Forest was selected because it performs well on tabular classification problems and provides feature importance.
- The evaluation metrics show that the model can classify breast cancer cases with strong performance.

The notebook is ready to be uploaded to GitHub and discussed by the four team members.


## GitHub Submission Structure

```text
Breast-Cancer-BigData-ML-Project/
│
├── Breast_Cancer_BigData_ML_Project_4Parts.ipynb
├── README.md
└── requirements.txt
```

The repository should include the final notebook and a short README explaining the project idea, dataset, tools, model, and results.
